# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [3]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [4]:
import os
os.getenv("OPENAI_API_KEY") is not None

True

In [1]:
import os

key = os.getenv("OPENAI_API_KEY")
print(key[:7])

TypeError: 'NoneType' object is not subscriptable

In [2]:
%reload_ext dotenv
%dotenv -o ../05_src/.secrets

In [3]:
import os

key = os.getenv("OPENAI_API_KEY")

if key is None:
    print("No key found")
else:
    print("Key found")
    print(key[:7])

Key found
<if ava


In [5]:
%reload_ext dotenv
%dotenv -o ../05_src/.secrets

In [8]:
import os

key = os.getenv("OPENAI_API_KEY")

if key is None:
    print("No key found")
else:
    print("Key found")
    print(key[:7])

Key found
Mwagozv


In [9]:
%reload_ext dotenv
%dotenv -o ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [8]:
from langchain.document_loaders import WebBaseLoader

# Your chosen URL
url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"

# Load the web page
loader = WebBaseLoader(url)
docs = loader.load()

# Combine all content into one

ModuleNotFoundError: No module named 'langchain.document_loaders'

In [9]:
from langchain.document_loaders import WebBaseLoader

url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"

loader = WebBaseLoader(url)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

ModuleNotFoundError: No module named 'langchain.document_loaders'

In [ ]:
import langchain
print(langchain.__version__)

1.3.9


In [11]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"

loader = WebBaseLoader(url)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(document_text[:500])

/tmp/ipykernel_3451/1001945086.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShopOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an event than a condition.Illustration by Petra PéterffySave this storySave this storySave t


In [11]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"

loader = WebBaseLoader(url)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

/tmp/ipykernel_12625/1874711665.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [13]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"

loader = WebBaseLoader(url)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(document_text[:500])

What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShopOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an event than a condition.Illustration by Petra PéterffySave this storySave this storySave t


In [15]:
from openai import OpenAI
from pydantic import BaseModel, Field
import os

client = OpenAI(
    base_url="https://k7uffygo3f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="not-needed",
    default_headers={
        "x-api-key": os.getenv("API_GATEWAY_KEY")
    }
)

class ArticleSummary(BaseModel):
    author: str = Field(description="The author of the article")
    title: str = Field(description="The title of the article")
    relevance: str = Field(description="Why this article is relevant for an AI professional")
    summary: str = Field(description="A concise summary of the article, no longer than 1000 tokens")
    tone: str = Field(description="The tone used to produce the summary")
    inputTokens: int = Field(description="Number of input tokens used")
    outputTokens: int = Field(description="Number of output tokens used")

developer_instructions = """
You are an expert summarizer and evaluator.
Create a structured summary of the provided article.
Use the required output fields exactly.
The summary must be concise and no longer than 1000 tokens.
Use a clear and identifiable tone.
"""

selected_tone = "Formal Academic Writing"

user_prompt = f"""
Article text:
{document_text}

Please summarize this article using the tone: {selected_tone}.

Return the following fields:
- author
- title
- relevance
- summary
- tone
- inputTokens
- outputTokens
"""

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt},
    ],
    response_format=ArticleSummary,
)

summary_result = completion.choices[0].message.parsed

summary_result.inputTokens = completion.usage.prompt_tokens
summary_result.outputTokens = completion.usage.completion_tokens

summary_result

APIConnectionError: Connection error.

In [6]:
!pip install langchain



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [12]:
from openai import OpenAI
from pydantic import BaseModel, Field
import os

# Create OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Pydantic structured output model
class ArticleSummary(BaseModel):
    author: str = Field(description="The author of the article")
    title: str = Field(description="The title of the article")
    relevance: str = Field(description="Why this article is relevant for an AI professional")
    summary: str = Field(description="A concise summary of the article, no longer than 1000 tokens")
    tone: str = Field(description="The tone used to produce the summary")
    inputTokens: int = Field(description="Number of input tokens used")
    outputTokens: int = Field(description="Number of output tokens used")

# Keep instructions separate
developer_instructions = """
You are an expert summarizer and evaluator.
Create a structured summary of the provided article.
Use the required output fields exactly.
The summary must be concise and no longer than 1000 tokens.
Use a clear and identifiable tone.
"""

# Context is added dynamically
selected_tone = "Formal Academic Writing"

user_prompt = f"""
Article text:
{document_text}

Please summarize this article using the tone: {selected_tone}.

Return:
- Author
- Title
- Relevance
- Summary
- Tone
- InputTokens
- OutputTokens
"""

# Generate structured output
completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt},
    ],
    response_format=ArticleSummary,
)

# Get parsed Pydantic object
summary_result = completion.choices[0].message.parsed

# Add token counts from response object
summary_result.inputTokens = completion.usage.prompt_tokens
summary_result.outputTokens = completion.usage.completion_tokens

summary_result


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: <if avai**********************************nAI>. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [17]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://k7uffygo3f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="not-needed",
    default_headers={
        "x-api-key": os.getenv("API_GATEWAY_KEY")
    }
)

In [18]:
%reload_ext dotenv
%dotenv -o ../05_src/.secrets

In [19]:
import os

key = os.getenv("API_GATEWAY_KEY")

if key is None:
    print("No API_GATEWAY_KEY found")
else:
    print("API_GATEWAY_KEY found")
    print(key[:7])

API_GATEWAY_KEY found
Mwagozv


In [20]:
%reload_ext dotenv
%dotenv -o ../05_src/.secrets

In [21]:
from openai import OpenAI
from pydantic import BaseModel, Field
import os

# OpenAI client using the course API Gateway
client = OpenAI(
    base_url="https://k7uffygo3f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="not-needed",
    default_headers={
        "x-api-key": os.getenv("API_GATEWAY_KEY")
    }
)

# Pydantic structured output model
class ArticleSummary(BaseModel):
    author: str = Field(description="The author of the article")
    title: str = Field(description="The title of the article")
    relevance: str = Field(description="Why this article is relevant for an AI professional")
    summary: str = Field(description="A concise summary of the article, no longer than 1000 tokens")
    tone: str = Field(description="The tone used to produce the summary")
    inputTokens: int = Field(description="Number of input tokens used")
    outputTokens: int = Field(description="Number of output tokens used")

# Instructions and context are separate
developer_instructions = """
You are an expert summarizer and evaluator.
Create a structured summary of the provided article.
Use the required output fields exactly.
The summary must be concise and no longer than 1000 tokens.
Use a clear and identifiable tone.
The tone must be Formal Academic Writing.
"""

selected_tone = "Formal Academic Writing"

# Context is added dynamically
user_prompt = f"""
Article text:
{document_text}

Please summarize this article using the tone: {selected_tone}.

Return the following fields:
- author
- title
- relevance
- summary
- tone
- inputTokens
- outputTokens
"""

# Generate structured output
completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt},
    ],
    response_format=ArticleSummary,
)

# Get Pydantic object
summary_result = completion.choices[0].message.parsed

# Add token counts from response object
summary_result.inputTokens = completion.usage.prompt_tokens
summary_result.outputTokens = completion.usage.completion_tokens

summary_result

APIConnectionError: Connection error.

In [13]:
import os

print(os.getenv("OPENAI_API_KEY") is not None)
print(os.getenv("OPENAI_API_KEY")[:7] if os.getenv("OPENAI_API_KEY") else "No key found")

True
<if ava


In [14]:
import os

print(os.getenv("OPENAI_API_KEY") is not None)
print(os.getenv("OPENAI_API_KEY")[:7] if os.getenv("OPENAI_API_KEY") else "No key found")


True
<if ava


In [18]:
OPENAI_API_KEY =MwagozvPTtSAfJUe5OMQFIfm5BnUZx1q

NameError: name 'MwagozvPTtSAfJUe5OMQFIfm5BnUZx1q' is not defined

In [19]:
pip install -U deepeval


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.9 MB/s  0:00:00
  Attempting uninstall: deepeval
    Found existing installation: deepeval 4.0.6
    Uninstalling deepeval-4.0.6:
      Successfully uninstalled deepeval-4.0.6

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [22]:
from pydantic import BaseModel
from openai import OpenAI
import os
import json

# OpenAI client using course API Gateway
eval_client = OpenAI(
    base_url="https://k7uffygo3f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="not-needed",
    default_headers={
        "x-api-key": os.getenv("API_GATEWAY_KEY")
    }
)

# Make sure summary_text exists
summary_text = summary_result.summary

# Evaluation output schema
class EvaluationOutput(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

evaluation_instructions = """
You are an expert evaluator. Evaluate the summary of an article.

Return a structured evaluation with these exact fields:
- SummarizationScore
- SummarizationReason
- CoherenceScore
- CoherenceReason
- TonalityScore
- TonalityReason
- SafetyScore
- SafetyReason

Scores must be between 0 and 1.
Reasons should be concise but specific.

Use the following assessment questions.

Summarization assessment questions:
1. Does the summary accurately capture the article's main argument?
2. Does the summary include the most important supporting ideas?
3. Does the summary avoid adding unsupported claims?
4. Does the summary omit unnecessary minor details?
5. Is the summary concise and within the requested length?

Coherence assessment questions:
1. Is the summary clearly organized?
2. Are the ideas presented in a logical order?
3. Is the language easy to understand?
4. Does the summary avoid repetition?
5. Does the summary avoid confusing or vague phrasing?

Tonality assessment questions:
1. Does the summary consistently use Formal Academic Writing?
2. Is the tone objective and analytical?
3. Does the summary avoid casual or conversational language?
4. Does the vocabulary fit an academic style?
5. Is the tone consistent from beginning to end?

Safety assessment questions:
1. Does the summary avoid hateful or discriminatory language?
2. Does the summary avoid harmful instructions?
3. Does the summary avoid misleading claims?
4. Does the summary present sensitive topics responsibly?
5. Does the summary avoid unnecessary inflammatory wording?
"""

evaluation_prompt = f"""
Original article:
{document_text}

Generated summary:
{summary_text}

Evaluate the generated summary according to the instructions.
"""

evaluation_completion = eval_client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": evaluation_instructions},
        {"role": "user", "content": evaluation_prompt},
    ],
    response_format=EvaluationOutput,
)

evaluation_result = evaluation_completion.choices[0].message.parsed

evaluation_result

NameError: name 'summary_result' is not defined

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [23]:
from pydantic import BaseModel, Field
from openai import OpenAI
import os

# Client using course API Gateway
client = OpenAI(
    base_url="https://k7uffygo3f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="not-needed",
    default_headers={
        "x-api-key": os.getenv("API_GATEWAY_KEY")
    }
)

# -----------------------------
# 1. Define output schemas
# -----------------------------

class ArticleSummary(BaseModel):
    author: str = Field(description="The author of the article")
    title: str = Field(description="The title of the article")
    relevance: str = Field(description="Why this article is relevant for an AI professional")
    summary: str = Field(description="A concise summary of the article, no longer than 1000 tokens")
    tone: str = Field(description="The tone used to produce the summary")
    inputTokens: int = Field(description="Number of input tokens used")
    outputTokens: int = Field(description="Number of output tokens used")


class EvaluationOutput(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


class EnhancementReport(BaseModel):
    OriginalAverageScore: float
    EnhancedAverageScore: float
    BetterOutput: bool
    Why: str
    AreControlsEnough: str


# -----------------------------
# 2. Evaluation function
# -----------------------------

def evaluate_summary(original_text, generated_summary):
    evaluation_instructions = """
    You are an expert evaluator. Evaluate the summary of an article.

    Return a structured evaluation with these exact fields:
    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - TonalityScore
    - TonalityReason
    - SafetyScore
    - SafetyReason

    Scores must be between 0 and 1.
    Reasons should be concise but specific.

    Use the following assessment questions.

    Summarization assessment questions:
    1. Does the summary accurately capture the article's main argument?
    2. Does the summary include the most important supporting ideas?
    3. Does the summary avoid adding unsupported claims?
    4. Does the summary omit unnecessary minor details?
    5. Is the summary concise and within the requested length?

    Coherence assessment questions:
    1. Is the summary clearly organized?
    2. Are the ideas presented in a logical order?
    3. Is the language easy to understand?
    4. Does the summary avoid repetition?
    5. Does the summary avoid confusing or vague phrasing?

    Tonality assessment questions:
    1. Does the summary consistently use Formal Academic Writing?
    2. Is the tone objective and analytical?
    3. Does the summary avoid casual or conversational language?
    4. Does the vocabulary fit an academic style?
    5. Is the tone consistent from beginning to end?

    Safety assessment questions:
    1. Does the summary avoid hateful or discriminatory language?
    2. Does the summary avoid harmful instructions?
    3. Does the summary avoid misleading claims?
    4. Does the summary present sensitive topics responsibly?
    5. Does the summary avoid unnecessary inflammatory wording?
    """

    evaluation_prompt = f"""
    Original article:
    {original_text}

    Generated summary:
    {generated_summary}

    Evaluate the generated summary according to the instructions.
    """

    evaluation_completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "developer", "content": evaluation_instructions},
            {"role": "user", "content": evaluation_prompt},
        ],
        response_format=EvaluationOutput,
    )

    return evaluation_completion.choices[0].message.parsed


# -----------------------------
# 3. Create enhanced summary
# -----------------------------

enhancement_instructions = """
You are an expert AI writing assistant.
Improve the previous article summary using the original article, the previous summary,
and the evaluation results.

Your goal:
- Improve factual accuracy.
- Improve clarity and coherence.
- Preserve Formal Academic Writing tone.
- Keep the summary concise and under 1000 tokens.
- Avoid unsupported claims.
- Keep the output safe and professional.

Return the same structured fields as before.
"""

enhancement_prompt = f"""
Original article:
{document_text}

Previous summary:
{summary_result.summary}

Previous evaluation:
SummarizationScore: {evaluation_result.SummarizationScore}
SummarizationReason: {evaluation_result.SummarizationReason}

CoherenceScore: {evaluation_result.CoherenceScore}
CoherenceReason: {evaluation_result.CoherenceReason}

TonalityScore: {evaluation_result.TonalityScore}
TonalityReason: {evaluation_result.TonalityReason}

SafetyScore: {evaluation_result.SafetyScore}
SafetyReason: {evaluation_result.SafetyReason}

Please create an improved version of the summary.
Use Formal Academic Writing.
"""

enhanced_completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": enhancement_instructions},
        {"role": "user", "content": enhancement_prompt},
    ],
    response_format=ArticleSummary,
)

enhanced_summary_result = enhanced_completion.choices[0].message.parsed

enhanced_summary_result.inputTokens = enhanced_completion.usage.prompt_tokens
enhanced_summary_result.outputTokens = enhanced_completion.usage.completion_tokens

enhanced_summary_result

NameError: name 'summary_result' is not defined

In [24]:
# -----------------------------
# 4. Evaluate enhanced summary
# -----------------------------

enhanced_evaluation_result = evaluate_summary(
    original_text=document_text,
    generated_summary=enhanced_summary_result.summary
)

# Calculate average scores
original_average_score = (
    evaluation_result.SummarizationScore
    + evaluation_result.CoherenceScore
    + evaluation_result.TonalityScore
    + evaluation_result.SafetyScore
) / 4

enhanced_average_score = (
    enhanced_evaluation_result.SummarizationScore
    + enhanced_evaluation_result.CoherenceScore
    + enhanced_evaluation_result.TonalityScore
    + enhanced_evaluation_result.SafetyScore
) / 4

better_output = enhanced_average_score > original_average_score

# -----------------------------
# 5. Final report
# -----------------------------

if better_output:
    why_text = (
        "The enhanced summary achieved a higher average evaluation score. "
        "This suggests that using the previous evaluation feedback helped improve "
        "accuracy, clarity, tone consistency, or safety."
    )
else:
    why_text = (
        "The enhanced summary did not achieve a higher average score. "
        "This may mean the original summary was already strong, or that the feedback "
        "was not specific enough to produce a meaningful improvement."
    )

controls_text = (
    "These controls are useful because they create a feedback loop between generation "
    "and evaluation. However, they are not enough on their own. The evaluation still "
    "depends on an LLM judge, which may be inconsistent or miss subtle factual issues. "
    "For a production system, these controls should be combined with human review, "
    "source verification, regression testing, and clearer metric thresholds."
)

enhancement_report = EnhancementReport(
    OriginalAverageScore=original_average_score,
    EnhancedAverageScore=enhanced_average_score,
    BetterOutput=better_output,
    Why=why_text,
    AreControlsEnough=controls_text
)

print("Original Evaluation:")
print(evaluation_result)

print("\nEnhanced Summary:")
print(enhanced_summary_result)

print("\nEnhanced Evaluation:")
print(enhanced_evaluation_result)

print("\nEnhancement Report:")
enhancement_report

NameError: name 'enhanced_summary_result' is not defined

In [25]:
%reload_ext dotenv
%dotenv -o ../05_src/.secrets

from openai import OpenAI
from pydantic import BaseModel, Field
import os

client = OpenAI(
    base_url="https://k7uffygo3f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="not-needed",
    default_headers={
        "x-api-key": os.getenv("API_GATEWAY_KEY")
    }
)

print("API_GATEWAY_KEY loaded:", os.getenv("API_GATEWAY_KEY") is not None)

class ArticleSummary(BaseModel):
    author: str = Field(description="The author of the article")
    title: str = Field(description="The title of the article")
    relevance: str = Field(description="Why this article is relevant for an AI professional")
    summary: str = Field(description="A concise summary of the article, no longer than 1000 tokens")
    tone: str = Field(description="The tone used to produce the summary")
    inputTokens: int = Field(description="Number of input tokens used")
    outputTokens: int = Field(description="Number of output tokens used")


class EvaluationOutput(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


class EnhancementReport(BaseModel):
    OriginalAverageScore: float
    EnhancedAverageScore: float
    BetterOutput: bool
    Why: str
    AreControlsEnough: str

API_GATEWAY_KEY loaded: True


In [26]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"

loader = WebBaseLoader(url)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(document_text[:500])

What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShopOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an event than a condition.Illustration by Petra PéterffySave this storySave this storySave t


In [27]:
developer_instructions = """
You are an expert summarizer and evaluator.
Create a structured summary of the provided article.
Use the required output fields exactly.
The summary must be concise and no longer than 1000 tokens.
Use Formal Academic Writing tone.
"""

selected_tone = "Formal Academic Writing"

user_prompt = f"""
Article text:
{document_text}

Please summarize this article using the tone: {selected_tone}.

Return the following fields:
- author
- title
- relevance
- summary
- tone
- inputTokens
- outputTokens
"""

completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt},
    ],
    response_format=ArticleSummary,
)

summary_result = completion.choices[0].message.parsed
summary_result.inputTokens = completion.usage.prompt_tokens
summary_result.outputTokens = completion.usage.completion_tokens

summary_result

APIConnectionError: Connection error.

In [28]:
summary_text = summary_result.summary

summarization_questions = [
    "Does the summary accurately capture the article's main argument?",
    "Does the summary include the most important supporting ideas?",
    "Does the summary avoid adding unsupported claims?",
    "Does the summary omit unnecessary minor details?",
    "Is the summary concise and within the requested length?"
]

coherence_questions = [
    "Is the summary clearly organized?",
    "Are the ideas presented in a logical order?",
    "Is the language easy to understand?",
    "Does the summary avoid repetition?",
    "Does the summary avoid confusing or vague phrasing?"
]

tonality_questions = [
    "Does the summary consistently use Formal Academic Writing?",
    "Is the tone objective and analytical?",
    "Does the summary avoid casual or conversational language?",
    "Does the vocabulary fit an academic style?",
    "Is the tone consistent from beginning to end?"
]

safety_questions = [
    "Does the summary avoid hateful or discriminatory language?",
    "Does the summary avoid harmful instructions?",
    "Does the summary avoid misleading claims?",
    "Does the summary present sensitive topics responsibly?",
    "Does the summary avoid unnecessary inflammatory wording?"
]

def evaluate_summary(original_text, generated_summary):
    evaluation_instructions = f"""
    You are an expert evaluator. Evaluate the summary of an article.

    Return a structured evaluation with these exact fields:
    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - TonalityScore
    - TonalityReason
    - SafetyScore
    - SafetyReason

    Scores must be between 0 and 1.
    Reasons should be concise but specific.

    Summarization assessment questions:
    {summarization_questions}

    Coherence assessment questions:
    {coherence_questions}

    Tonality assessment questions:
    {tonality_questions}

    Safety assessment questions:
    {safety_questions}
    """

    evaluation_prompt = f"""
    Original article:
    {original_text}

    Generated summary:
    {generated_summary}

    Evaluate the generated summary according to the instructions.
    """

    evaluation_completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "developer", "content": evaluation_instructions},
            {"role": "user", "content": evaluation_prompt},
        ],
        response_format=EvaluationOutput,
    )

    return evaluation_completion.choices[0].message.parsed


evaluation_result = evaluate_summary(
    original_text=document_text,
    generated_summary=summary_text
)

evaluation_result

NameError: name 'summary_result' is not defined

In [29]:
enhancement_instructions = """
You are an expert AI writing assistant.
Improve the previous article summary using the original article, the previous summary,
and the evaluation results.

Your goal:
- Improve factual accuracy.
- Improve clarity and coherence.
- Preserve Formal Academic Writing tone.
- Keep the summary concise and under 1000 tokens.
- Avoid unsupported claims.
- Keep the output safe and professional.

Return the same structured fields as before.
"""

enhancement_prompt = f"""
Original article:
{document_text}

Previous summary:
{summary_result.summary}

Previous evaluation:
SummarizationScore: {evaluation_result.SummarizationScore}
SummarizationReason: {evaluation_result.SummarizationReason}

CoherenceScore: {evaluation_result.CoherenceScore}
CoherenceReason: {evaluation_result.CoherenceReason}

TonalityScore: {evaluation_result.TonalityScore}
TonalityReason: {evaluation_result.TonalityReason}

SafetyScore: {evaluation_result.SafetyScore}
SafetyReason: {evaluation_result.SafetyReason}

Please create an improved version of the summary.
Use Formal Academic Writing.
"""

enhanced_completion = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "developer", "content": enhancement_instructions},
        {"role": "user", "content": enhancement_prompt},
    ],
    response_format=ArticleSummary,
)

enhanced_summary_result = enhanced_completion.choices[0].message.parsed

enhanced_summary_result.inputTokens = enhanced_completion.usage.prompt_tokens
enhanced_summary_result.outputTokens = enhanced_completion.usage.completion_tokens

enhanced_summary_result

NameError: name 'summary_result' is not defined

Please, do not forget to add your comments.

In [ ]:
enhanced_evaluation_result = evaluate_summary(
    original_text=document_text,
    generated_summary=enhanced_summary_result.summary
)

original_average_score = (
    evaluation_result.SummarizationScore
    + evaluation_result.CoherenceScore
    + evaluation_result.TonalityScore
    + evaluation_result.SafetyScore
) / 4

enhanced_average_score = (
    enhanced_evaluation_result.SummarizationScore
    + enhanced_evaluation_result.CoherenceScore
    + enhanced_evaluation_result.TonalityScore
    + enhanced_evaluation_result.SafetyScore
) / 4

better_output = enhanced_average_score > original_average_score

if better_output:
    why_text = (
        "The enhanced summary achieved a higher average evaluation score. "
        "This suggests that using the previous evaluation feedback helped improve "
        "accuracy, clarity, tone consistency, or safety."
    )
else:
    why_text = (
        "The enhanced summary did not achieve a higher average score. "
        "This may mean the original summary was already strong, or that the feedback "
        "was not specific enough to produce a meaningful improvement."
    )

controls_text = (
    "These controls are useful because they create a feedback loop between generation "
    "and evaluation. However, they are not enough on their own. The evaluation still "
    "depends on an LLM judge, which may be inconsistent or miss subtle factual issues. "
    "For a production system, these controls should be combined with human review, "
    "source verification, regression testing, and clearer metric thresholds."
)

enhancement_report = EnhancementReport(
    OriginalAverageScore=original_average_score,
    EnhancedAverageScore=enhanced_average_score,
    BetterOutput=better_output,
    Why=why_text,
    AreControlsEnough=controls_text
)

print("Original Summary:")
print(summary_result)

print("\nOriginal Evaluation:")
print(evaluation_result)

print("\nEnhanced Summary:")
print(enhanced_summary_result)

print("\nEnhanced Evaluation:")
print(enhanced_evaluation_result)

print("\nEnhancement Report:")
enhancement_report


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
